# 1. Import Libraries

In [9]:
!pip install transformers pandas torch
!pip install 'accelerate>=1.1.0'
!pip install scikit-learn
!pip install datasets


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [10]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import accelerate
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
import numpy as np
from sklearn.metrics import classification_report

# 2. Import Dataset and Train-test Split

In [11]:
df = pd.read_csv("../dataset/dataset.csv")

labels = ["Negatif", "Positif"]

def encode_sentiment(dataframe):
    dataframe["labels"] = dataframe["sentiment"].apply(labels.index)
    return dataframe

def decode_sentiment(dataframe):
    dataframe["sentiment_decoded"] = dataframe["labels"].apply(lambda x: labels[x])
    return dataframe

df = encode_sentiment(df)


total_reviews = df["review_id"].max()

train_percentage = 0.8

max_id_train = total_reviews * train_percentage

train_df = df[df["review_id"] <= max_id_train]
test_df = df[df["review_id"] > max_id_train]

train_df

,url,judul,review,sentiment,review_id,labels
0,https://www.cnbcindonesia.com/research/2025012...,Trump Sebar Exceutive Order: Emang Semengerika...,"Jakarta, CNBC Indonesia -Amerika Serikat (AS) ...",Positif,1,1
1,https://www.cnbcindonesia.com/research/2025012...,"Alasan Rupiah 'Berpesta' di Pelantikan Trump, ...","Jakarta, CNBC Indonesia -Nilai tukar rupiah te...",Positif,2,1
2,https://www.cnbcindonesia.com/research/2025012...,"Trump Beri Kabar Baik, Saatnya Menunggu Dolar ...","Jakarta, CNBC Indonesia-Pasar keuangan Indones...",Positif,3,1
3,https://www.cnbcindonesia.com/research/2025030...,"IHSG Merah Lagi, Begini Penjelasan dari Analis...","Jakarta, CNBC Indonesia -Indeks Harga Saham Ga...",Negatif,4,0
4,https://indodax.com/academy/bitcoin-200k-predi...,Bernstein: Bitcoin Bisa Naik 2x Lipat! Target ...,HargaBitcoin(BTC)pernah melewati angka terting...,Positif,5,1
...,...,...,...,...,...,...
795,https://www.cnbcindonesia.com/tech/20250506105...,"Tarif Trump Sia-sia, Ini Alasan China Tak Bisa...","Jakarta, CNBC Indonesia -Usaha Trump untuk mem...",Negatif,796,0
796,https://megapolitan.antaranews.com/berita/3932...,Tarif Trump dan ujian diplomasi Indonesia - AN...,"Senin, 5 Mei 2025 13:31 WIB\nPemerintah Indone...",Positif,797,1
797,https://www.bloombergtechnoz.com/detail-news/7...,Donald Trump Isyaratkan akan Turunkan Tarif Im...,"Kasia Klimasinska - Bloomberg News\nBloomberg,...",Positif,798,1
798,https://www.antaranews.com/berita/4817485/bea-...,Bea Cukai: Penundaan tarif Trump pacu produksi...,Batam (ANTARA) - Kepala Kantor Pelayanan Utama...,Positif,799,1


# 3. Import IndoBERT

In [12]:
model_name = "indobenchmark/indobert-base-p2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36168.76it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 4. Fine Tune Model

In [13]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize(examples):
    return tokenizer(
        examples["review"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset

Map: 100%|██████████| 201/201 [00:00<00:00, 1891.19 examples/s]


Dataset({
    features: ['url', 'judul', 'review', 'sentiment', 'review_id', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 800
})

In [14]:
training_args = TrainingArguments(
    output_dir="./results2",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]


TrainOutput(global_step=250, training_loss=0.3735402221679687, metrics={'train_runtime': 111.566, 'train_samples_per_second': 35.853, 'train_steps_per_second': 2.241, 'total_flos': 263111055360000.0, 'train_loss': 0.3735402221679687, 'epoch': 5.0})

# 5. Predict

In [15]:
predictions = trainer.predict(test_dataset)
pred_ids = np.argmax(predictions.predictions, axis=1)

predictions_df = pd.DataFrame({
    "prediction": [labels[i] for i in pred_ids]
})

predictions_df["actuals"] = test_df.reset_index(drop=True)["sentiment"]
predictions_df["review_id"] = test_df.reset_index(drop=True)["review_id"]
predictions_df["sentiment"] = test_df.reset_index(drop=True)["sentiment"]
predictions_df

,prediction,actuals,review_id,sentiment
0,Negatif,Positif,801,Positif
1,Positif,Positif,802,Positif
2,Negatif,Positif,803,Positif
3,Positif,Positif,804,Positif
4,Positif,Positif,805,Positif
...,...,...,...,...
196,Positif,Positif,997,Positif
197,Negatif,Positif,998,Positif
198,Negatif,Negatif,999,Negatif
199,Positif,Negatif,1000,Negatif


# 6. Evaluate

In [16]:
print(
    classification_report(
        test_df["labels"],
        pred_ids,
        target_names=["Negatif", "Positif"]
    )
)

              precision    recall  f1-score   support

     Negatif       0.50      0.82      0.62        72
     Positif       0.84      0.53      0.65       129

    accuracy                           0.64       201
   macro avg       0.67      0.68      0.64       201
weighted avg       0.72      0.64      0.64       201



# 